# ⚙️ Feature Engineering - Telco Customer Churn

**Objective:** Create, transform, and prepare features for machine learning models.

**Tasks:**
- Feature encoding (categorical → numerical)
- Feature scaling (normalization/standardization)
- Feature creation (new derived features)
- Feature selection
- Train/test split

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing tools
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
print("✅ Libraries imported successfully!")

## 2. Load Cleaned Dataset

In [ ]:
# Load cleaned data from previous notebook
df = pd.read_csv('../artifacts/data/telco_churn_cleaned.csv')

print(f"Dataset loaded: {df.shape}")
df.head()

In [ ]:
# Make a copy for feature engineering
df_fe = df.copy()
print(f"Working copy created: {df_fe.shape}")

## 3. Separate Features and Target

In [ ]:
# Identify feature and target columns
target_col = 'Churn'
id_col = 'customerID'

# Store customer IDs separately
customer_ids = df_fe[id_col].copy()

# Separate target
y = df_fe[target_col].copy()

# Drop ID and target from features
X = df_fe.drop([id_col, target_col], axis=1)

print(f"\n📊 Dataset split:")
print(f"   Features (X): {X.shape}")
print(f"   Target (y): {y.shape}")
print(f"   Customer IDs: {len(customer_ids)}")

In [ ]:
# Identify feature types
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\n🔤 Categorical features ({len(categorical_cols)}):")
print(categorical_cols)
print(f"\n🔢 Numerical features ({len(numerical_cols)}):")
print(numerical_cols)

## 4. Feature Creation (New Features)

In [ ]:
print("\n⚙️ Creating new features...")
print("="*60)

# 1. Tenure group (categorize customer lifecycle)
X['tenure_group'] = pd.cut(X['tenure'], 
                           bins=[0, 12, 24, 48, 72], 
                           labels=['0-1 year', '1-2 years', '2-4 years', '4+ years'])

# 2. Average charges per month
X['avg_charge_per_tenure'] = X['TotalCharges'] / (X['tenure'] + 1)  # +1 to avoid division by zero

# 3. Service usage score (count of services)
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']

X['service_count'] = 0
for col in service_cols:
    if col in X.columns:
        X['service_count'] += (X[col] == 'Yes').astype(int)

# 4. Has phone service (binary)
X['has_phone'] = (X['PhoneService'] == 'Yes').astype(int)

# 5. Has internet service (binary)
X['has_internet'] = (X['InternetService'] != 'No').astype(int)

# 6. Has premium services (security, backup, protection)
premium_services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection']
X['premium_services_count'] = 0
for col in premium_services:
    if col in X.columns:
        X['premium_services_count'] += (X[col] == 'Yes').astype(int)

# 7. Has streaming services
streaming_cols = ['StreamingTV', 'StreamingMovies']
X['streaming_services_count'] = 0
for col in streaming_cols:
    if col in X.columns:
        X['streaming_services_count'] += (X[col] == 'Yes').astype(int)

# 8. Is senior with dependents
X['senior_with_dependents'] = ((X['SeniorCitizen'] == 'Yes') & (X['Dependents'] == 'Yes')).astype(int)

# 9. Monthly to total charges ratio
X['monthly_to_total_ratio'] = X['MonthlyCharges'] / (X['TotalCharges'] + 1)

# 10. Contract type encoding (ordinal: month-to-month < one year < two year)
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
X['contract_encoded'] = X['Contract'].map(contract_map)

print("✅ Created 10+ new features!")
print(f"\nNew feature shape: {X.shape}")

In [ ]:
# Display sample of new features
new_features = ['tenure_group', 'avg_charge_per_tenure', 'service_count', 
                'premium_services_count', 'streaming_services_count', 'monthly_to_total_ratio']
print("\n📊 Sample of new features:")
X[new_features].head(10)

## 5. Feature Encoding

In [ ]:
# Update categorical columns list (after adding new features)
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\n🔤 Categorical features to encode: {len(categorical_cols)}")
print(categorical_cols[:10])  # Show first 10

In [ ]:
# Label Encoding for binary categorical features (Yes/No)
print("\n⚙️ Encoding binary categorical features...")

binary_cols = []
for col in categorical_cols:
    unique_values = X[col].nunique()
    if unique_values == 2:
        binary_cols.append(col)

print(f"Binary columns: {len(binary_cols)}")

# Create label encoders
label_encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    print(f"  • {col}: {le.classes_}")

print("\n✅ Binary encoding completed!")

In [ ]:
# One-Hot Encoding for multi-class categorical features
# Update categorical list after label encoding
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_cols) > 0:
    print(f"\n⚙️ One-hot encoding for multi-class features: {len(categorical_cols)}")
    print(categorical_cols)
    
    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    
    print(f"\n✅ One-hot encoding completed!")
    print(f"   Before: {X.shape[1]} features")
    print(f"   After: {X_encoded.shape[1]} features")
    
    X = X_encoded
else:
    print("\n✅ No multi-class features to encode")

In [ ]:
# Encode target variable
print("\n⚙️ Encoding target variable...")
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f"Target classes: {le_target.classes_}")
print(f"Encoding: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
print(f"\nTarget distribution after encoding:")
print(pd.Series(y_encoded).value_counts().sort_index())

## 6. Feature Scaling

In [ ]:
# Check current feature ranges
print("\n📊 Feature ranges before scaling:")
print(X.describe().T[['min', 'max', 'mean', 'std']].head(10))

In [ ]:
# StandardScaler (z-score normalization)
print("\n⚙️ Applying StandardScaler...")
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("✅ Scaling completed!")
print(f"\n📊 Feature ranges after scaling:")
print(X_scaled.describe().T[['min', 'max', 'mean', 'std']].head(10))

In [ ]:
# Visualize scaling effect
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Before scaling
X[['tenure', 'MonthlyCharges', 'TotalCharges']].boxplot(ax=axes[0])
axes[0].set_title('Before Scaling', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Value')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)

# After scaling
X_scaled[['tenure', 'MonthlyCharges', 'TotalCharges']].boxplot(ax=axes[1])
axes[1].set_title('After Scaling (StandardScaler)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Value')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Quick feature importance using correlation with target
from sklearn.feature_selection import mutual_info_classif

print("\n🔍 Calculating feature importance...")

# Calculate mutual information
mi_scores = mutual_info_classif(X_scaled, y_encoded, random_state=42)
mi_scores = pd.Series(mi_scores, index=X_scaled.columns).sort_values(ascending=False)

print("\n📊 Top 15 Most Important Features:")
print(mi_scores.head(15))

In [ ]:
# Visualize feature importance
plt.figure(figsize=(12, 8))
mi_scores.head(20).sort_values().plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Top 20 Feature Importance (Mutual Information)', fontsize=16, fontweight='bold')
plt.xlabel('Mutual Information Score')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

## 8. Train-Test Split

In [ ]:
# Split data into train and test sets
print("\n🔀 Splitting data into train and test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded  # Maintain class distribution
)

print(f"\n✅ Data split completed!")
print(f"\n📊 Split Summary:")
print(f"   Training set: {X_train.shape}")
print(f"   Test set: {X_test.shape}")
print(f"   Split ratio: {len(X_train)/len(X_scaled)*100:.1f}% train, {len(X_test)/len(X_scaled)*100:.1f}% test")

In [ ]:
# Verify class distribution
print("\n📊 Class Distribution:")
print("\nOriginal:")
print(pd.Series(y_encoded).value_counts(normalize=True))
print("\nTraining set:")
print(pd.Series(y_train).value_counts(normalize=True))
print("\nTest set:")
print(pd.Series(y_test).value_counts(normalize=True))

## 9. Save Processed Data

In [ ]:
import os
import pickle

# Create directory
os.makedirs('../artifacts/data', exist_ok=True)

print("\n💾 Saving processed datasets...")

# Save train/test splits
X_train.to_csv('../artifacts/data/X_train.csv', index=False)
X_test.to_csv('../artifacts/data/X_test.csv', index=False)
pd.DataFrame(y_train, columns=['Churn']).to_csv('../artifacts/data/y_train.csv', index=False)
pd.DataFrame(y_test, columns=['Churn']).to_csv('../artifacts/data/y_test.csv', index=False)

print("✅ Train/test splits saved!")

# Save encoders and scaler
with open('../artifacts/data/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

with open('../artifacts/data/target_encoder.pkl', 'wb') as f:
    pickle.dump(le_target, f)

with open('../artifacts/data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Encoders and scaler saved!")

# Save feature names
feature_names = {'feature_names': X_scaled.columns.tolist()}
with open('../artifacts/data/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("✅ Feature names saved!")

## 10. Feature Engineering Summary

In [ ]:
print("\n" + "="*70)
print("📋 FEATURE ENGINEERING SUMMARY")
print("="*70)

print("\n✅ COMPLETED TASKS:")
print("   • Created 10+ new features")
print("   • Encoded categorical variables (Label + One-Hot)")
print("   • Scaled numerical features (StandardScaler)")
print("   • Analyzed feature importance")
print("   • Split data (80/20 train/test)")
print("   • Saved processed datasets and transformers")

print("\n📊 FINAL FEATURE SET:")
print(f"   • Total features: {X_scaled.shape[1]}")
print(f"   • Training samples: {X_train.shape[0]:,}")
print(f"   • Test samples: {X_test.shape[0]:,}")
print(f"   • Churn rate: {y_encoded.mean()*100:.2f}%")

print("\n🎯 READY FOR MODELING!")
print("   Next: 04_model_experiments.ipynb")
print("\n" + "="*70)